In [2]:
from langchain.document_loaders import PyPDFLoader

# Load a PDF file
loader = PyPDFLoader("meidtations.pdf")
documents = loader.load()

# View a small sample
print(documents[0].page_content[:300])


Ignoring wrong pointing object 6 0 (offset 0)
Ignoring wrong pointing object 8 0 (offset 0)
Ignoring wrong pointing object 12 0 (offset 0)
Ignoring wrong pointing object 15 0 (offset 0)
Ignoring wrong pointing object 24 0 (offset 0)
Ignoring wrong pointing object 26 0 (offset 0)


Marcus Aurelius' Meditations - tr. Casaubon v. 8.16, uploaded to www.philaletheians.co.uk, 14 July 2013 
Page 1 of 128 
The meditations of 
Marcus Aurelius Antoninus 
Originally translated by Meric Casaubon 
 
About this edition 
Marcus Aurelius Antoninus Augustus was Emperor of Rome from 161 to his


In [3]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

# documents = ...  # the list you loaded in Step 1

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,     # characters per chunk
    chunk_overlap=200,   # overlap to keep context flow
)

chunked_docs = splitter.split_documents(documents)
print(f"Total chunks: {len(chunked_docs)}")
print(chunked_docs[0].page_content[:250])


Total chunks: 576
Marcus Aurelius' Meditations - tr. Casaubon v. 8.16, uploaded to www.philaletheians.co.uk, 14 July 2013 
Page 1 of 128 
The meditations of 
Marcus Aurelius Antoninus 
Originally translated by Meric Casaubon 
 
About this edition 
Marcus Aurelius Anto


In [4]:
from langchain.vectorstores import FAISS
from langchain.embeddings import HuggingFaceEmbeddings

# Use a good all-rounder embedding model
embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# Embed and index the documents
vector_store = FAISS.from_documents(chunked_docs, embedding_model)

# Save locally (optional)
vector_store.save_local("rag_faiss_index")


/var/folders/p2/2t3fn50s5rv4wjpsrn3dj_780000gn/T/ipykernel_1047/3027492633.py:5: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
/Users/adithyachowdary/Desktop/rag/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/adithyachowdary/Desktop/rag/venv/lib/python3.12/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.

In [5]:
from langchain.vectorstores import FAISS
from langchain.embeddings import HuggingFaceEmbeddings

# Load your saved index
embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vector_store = FAISS.load_local("rag_faiss_index", embedding_model,allow_dangerous_deserialization=True)

# User's question
query = "What is this document about?"  # change as needed

# Retrieve top 3 relevant chunks
retrieved_docs = vector_store.similarity_search(query, k=3)

# Print retrieved results
for i, doc in enumerate(retrieved_docs, 1):
    print(f"\n--- Chunk {i} ---\n")
    print(doc.page_content[:500])



--- Chunk 1 ---

THE THIRD BOOK 23 
THE FOURTH BOOK 29 
THE FIFTH BOOK 38 
THE SIXTH BOOK 47 
THE SEVENTH BOOK 57 
THE EIGHTH BOOK 67 
THE NINTH BOOK 77 
THE TENTH BOOK 86 
THE ELEVENTH BOOK 96 
THE TWELFTH BOOK 104 
Appendix 110 
Notes 122 
Glossary 123 
A parting thought 128 
 
                                            2 [Brought forward from p. xxiii.]

--- Chunk 2 ---

Property of Pulchra, and another in which he impeaches a tribune. Ho, ho! I 
hear you cry to your man, Off with you as fast as you can, and bring me these 
speeches from the library of Apollo. No use to send: I have those books with me 
too. You must get round the Tiberian librarian; you will have to spend some-
thing on the matter; and when I return to town, I shall expect to go shares with 
him. Well, after reading these speeches I wrote a wretched trifle, destined for 
drowning or burning. No

--- Chunk 3 ---

“The Golden Book of Marcus Aurelius,” with an Introduction by W.H.D. Rouse. It was 
subsequently edite

In [6]:
from transformers import pipeline

# Use a local model (small & fast)
qa_pipeline = pipeline("question-answering", model="deepset/tinyroberta-squad2")

# Or for bigger answers, use: google/flan-t5-base with text2text-generation


Device set to use mps:0


In [7]:
# Combine the retrieved chunks into one context string
context = "\n\n".join([doc.page_content for doc in retrieved_docs])

# Ask a question
query = "What is meidtation ?"

# Run QA on context
answer = qa_pipeline(question=query, context=context)

print("Final Answer:\n")
print(answer["answer"])


Final Answer:

THE ORIGIN OF THE MYSTERIES


/Users/adithyachowdary/Desktop/rag/venv/lib/python3.12/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `RobertaSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)
